## Prepare imports

In [3]:
from datasets import get_test_loader, synsetid_to_cate
from args import get_parser
import torch
import pyvista as pv
from models.encoder_sparse import BaseModel as EctEncoder
import matplotlib.pyplot as plt 
from lightning.pytorch.utilities.model_summary import ModelSummary
from load_model_scaled import load_encoder, load_vae
# pv.set_jupyter_backend("static")

from model_wrapper import (
    TopologicalModelVAE,
    TopologicalModelEncoderSparse,
    ShapeNetModel,
    TopologicalModelEncoderScaled,
)

CATES = ["chair"]


In [4]:
from model_wrapper import TopologicalModelEncoder


encoder_model = EctEncoder.load_from_checkpoint(
    checkpoint_path=f"./trained_models/ectencoder_shapenet_{CATES[0]}_sparse.ckpt"
).cuda()
model = TopologicalModelEncoderSparse(encoder_model.cuda())

In [5]:
p = get_parser()
args, unknown = p.parse_known_args({"fast_run":False,"batch_size":8})
args.cates = CATES

In [6]:
print(args)

Namespace(input_dim=3, dims='256', latent_dims='256', num_blocks=1, latent_num_blocks=1, layer_type='concatsquash', time_length=0.5, train_T=True, nonlinearity='tanh', use_adjoint=True, solver='dopri5', atol=1e-05, rtol=1e-05, batch_norm=True, sync_bn=False, bn_lag=0, use_latent_flow=False, use_deterministic_encoder=False, zdim=128, optimizer='adam', batch_size=16, lr=0.001, beta1=0.9, beta2=0.999, momentum=0.9, weight_decay=0.0, epochs=100, seed=None, recon_weight=1.0, prior_weight=1.0, entropy_weight=1.0, scheduler='linear', exp_decay=1.0, exp_decay_freq=1, dataset_type='shapenet15k', cates=['chair'], data_dir='data/ShapeNetCore.v2.PC15k', mn40_data_dir='data/ModelNet40.PC15k', mn10_data_dir='data/ModelNet10.PC15k', dataset_scale=1.0, random_rotate=False, normalize_per_shape=False, normalize_std_per_axis=False, tr_max_sample_points=2048, te_max_sample_points=2048, num_workers=4, log_name=None, viz_freq=10, val_freq=10, log_freq=10, save_freq=10, no_validation=False, save_val_results=

In [7]:
# encoder_model = EctEncoder.load_from_checkpoint(
#     checkpoint_path=f"./trained_models/ectencoder_shapenet_{args.cates[0]}_scaled.ckpt").cuda()

In [8]:
# print(ModelSummary(encoder_model))
# model = TopologicalModelEncoderScaled(encoder_model)

In [9]:
# TODO: make this memory efficient
if "all" in args.cates:
    cates = list(synsetid_to_cate.values())
else:
    cates = args.cates
all_results = {}
cate_to_len = {}
for cate in cates:
    args.cates = [cate]
    loader = get_test_loader(args)

    all_sample = []
    all_ref = []
    for data in loader:
        idx_b, tr_pc, te_pc = data["idx"], data["train_points"], data["test_points"]

        te_pc = te_pc.cuda() if args.gpu is None else te_pc.cuda(args.gpu)
        tr_pc = tr_pc.cuda() if args.gpu is None else tr_pc.cuda(args.gpu)
        B, N = te_pc.size(0), te_pc.size(1)
        out_pc = model.reconstruct(tr_pc, num_points=N)

        m, s = data["mean"].float(), data["std"].float()
        m = m.cuda() if args.gpu is None else m.cuda(args.gpu)
        s = s.cuda() if args.gpu is None else s.cuda(args.gpu)
        out_pc = out_pc * s + m
        te_pc = te_pc * s + m

        all_sample.append(out_pc)
        all_ref.append(te_pc)

    sample_pcs = torch.cat(all_sample, dim=0)
    ref_pcs = torch.cat(all_ref, dim=0)
    cate_to_len[cate] = int(sample_pcs.size(0))


Total number of data:4612
Min number of points: (train)2048 (test)2048
Total number of data:662
Min number of points: (train)2048 (test)2048


In [10]:
jnt = torch.cat([sample_pcs,ref_pcs],axis=1)


In [12]:
pl = pv.Plotter(shape=(3,10), window_size=[1600, 600],border=False,polygon_smoothing=True)

offset=0
for col in range(10):
    points = ref_pcs[col+offset].reshape(-1, 3).detach().cpu().numpy()
    pl.subplot(0, col)
    actor = pl.add_points(
        points,
        style="points",
        emissive=False,
        show_scalar_bar=False,
        render_points_as_spheres=True,
        color="lightblue",
        point_size=5,
        ambient=0.2, 
        diffuse=0.8, 
        specular=0.8,
        specular_power=40, 
        smooth_shading=True
    )
    points = sample_pcs[col+offset].reshape(-1, 3).detach().cpu().numpy()
    pl.subplot(1, col)
    actor = pl.add_points(
        points,
        style="points",
        emissive=False,
        show_scalar_bar=False,
        render_points_as_spheres=True,
        color="lightblue",
        point_size=5,
        ambient=0.2, 
        diffuse=0.8, 
        specular=0.8,
        specular_power=40, 
        smooth_shading=True
    )
    points = jnt[col+offset].reshape(-1, 3).detach().cpu().numpy()
    pl.subplot(2, col)
    actor = pl.add_points(
        points,
        style="points",
        emissive=False,
        show_scalar_bar=False,
        render_points_as_spheres=True,
        color="lightblue",
        point_size=5,
        ambient=0.2, 
        diffuse=0.8, 
        specular=0.8,
        specular_power=40, 
        smooth_shading=True
    )


pl.background_color = "w"
pl.link_views()
pl.camera_position = "xy"
pos = pl.camera.position
# print(pos)
pl.camera.position = (pos[0],pos[1]+3,pos[2])
pl.camera.position = (3,0,0)
pl.camera.azimuth = 135
pl.camera.elevation = 30
# create a top down light
light = pv.Light(position=(0, 1, 0), positional=True,
                cone_angle=50, exponent=20, intensity=.2)
pl.add_light(light)
pl.camera.zoom(1)
pl.screenshot(f"./figures/pointcloud_encoder_{args.cates[0]}.png",transparent_background=True,scale=2)

pl.show()



Widget(value='<iframe src="http://localhost:53428/index.html?ui=P_0x1aea1d5a080_1&reconnect=auto" class="pyvis…

In [10]:
# pl = pv.Plotter(shape=(3,10), window_size=[1600, 600],border=False,polygon_smoothing=True)
# for offset in range(0,400,10):
#     for col in range(10):
#         points = ref_pcs[col+offset].reshape(-1, 3).detach().cpu().numpy()
#         pl.subplot(0, col)
#         actor = pl.add_points(
#             points,
#             style="points",
#             emissive=False,
#             show_scalar_bar=False,
#             render_points_as_spheres=True,
#             scalars=points[:, 2],
#             point_size=5,
#             ambient=0.2, 
#             diffuse=0.8, 
#             specular=0.8,
#             specular_power=40, 
#             smooth_shading=True
#         )
#         points = sample_pcs[col+offset].reshape(-1, 3).detach().cpu().numpy()
#         pl.subplot(1, col)
#         actor = pl.add_points(
#             points,
#             style="points",
#             emissive=False,
#             show_scalar_bar=False,
#             render_points_as_spheres=True,
#             scalars=points[:, 2],
#             point_size=5,
#             ambient=0.2, 
#             diffuse=0.8, 
#             specular=0.8,
#             specular_power=40, 
#             smooth_shading=True
#         )
#         points = jnt[col+offset].reshape(-1, 3).detach().cpu().numpy()
#         pl.subplot(2, col)
#         actor = pl.add_points(
#             points,
#             style="points",
#             emissive=False,
#             show_scalar_bar=False,
#             render_points_as_spheres=True,
#             scalars=points[:, 2],
#             point_size=5,
#             ambient=0.2, 
#             diffuse=0.8, 
#             specular=0.8,
#             specular_power=40, 
#             smooth_shading=True
#         )


#     pl.background_color = "w"
#     pl.link_views()
#     pl.camera_position = "yz"
#     pos = pl.camera.position
#     pl.camera.position = (pos[0],pos[1],pos[2]+3)
#     pl.camera.azimuth = -45
#     pl.camera.elevation = 10
#     # create a top down light
#     light = pv.Light(position=(0, 0, 3), positional=True,
#                     cone_angle=50, exponent=20, intensity=.2)
#     pl.add_light(light)
#     pl.camera.zoom(1.3)
#     pl.screenshot(f"./pointcloud_{offset}.png",transparent_background=True,scale=2)
#     pl.clear_actors()